# 🔬 ASSIGNMENT NLP – 4: Fine-Tuning BERT on IMDB Sentiment Dataset

**Internship:** Data Science Internship – February 2026  
**Task:** Fine-tune a pre-trained BERT model on the IMDB Movie Reviews dataset for binary sentiment classification.

---

## 📌 Objective
Fine-tune `bert-base-uncased` on the IMDB dataset, run controlled experiments (frozen layers vs fine-tuned layers), and evaluate using Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

---

## 🗂️ Table of Contents
1. [Install & Import Libraries](#1)
2. [Load & Explore Dataset](#2)
3. [Data Preprocessing](#3)
4. [Train / Validation / Test Split](#4)
5. [Tokenization with bert-base-uncased](#5)
6. [Build PyTorch Dataset & DataLoaders](#6)
7. [Model Building](#7)
8. [Training & Evaluation Utilities](#8)
9. [Experiment 1 — Freeze All BERT Layers (Classifier Only)](#9)
10. [Experiment 2 — Fine-Tune Last 2 BERT Layers](#10)
11. [Experiment 3 — Full BERT Fine-Tuning (Bonus)](#11)
12. [Results Comparison & Analysis](#12)
13. [Confusion Matrix Visualisation](#13)
14. [Insights & Conclusions](#14)

---
## 1. Install & Import Libraries <a id='1'></a>

In [ ]:
# Install required packages (run once)
!pip install transformers datasets torch scikit-learn matplotlib seaborn --quiet

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────

# Standard library
import re
import time
import warnings
warnings.filterwarnings("ignore")

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Hugging Face
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup      # Bonus: LR scheduler
)

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split

# ── Device setup ──────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Reproducibility seed ──────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("✅ All libraries imported successfully!")
print(f"   → PyTorch version : {torch.__version__}")
print(f"   → Device          : {DEVICE}")

---
## 2. Load & Explore Dataset <a id='2'></a>

**Dataset:** IMDB Movie Reviews (Hugging Face `datasets` library)  
- **50,000** labelled reviews (25k train / 25k test)  
- **Binary labels:** `0 = Negative`, `1 = Positive`  
- Widely used benchmark for sentiment analysis

> 💡 We use `datasets` to load IMDB directly — no manual Kaggle download needed. The dataset is identical to the Kaggle IMDB dataset.

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD DATASET
# ─────────────────────────────────────────────────────────────

print("⏳ Loading IMDB dataset...")
raw_dataset = load_dataset("imdb")

# Convert to pandas DataFrames for easier manipulation
train_df = pd.DataFrame(raw_dataset["train"])
test_df  = pd.DataFrame(raw_dataset["test"])

print(f"✅ Dataset loaded!")
print(f"   → Training samples : {len(train_df):,}")
print(f"   → Test samples     : {len(test_df):,}")
print(f"   → Columns          : {list(train_df.columns)}")

In [ ]:
# ── Explore the data ──────────────────────────────────────────

print("── Sample Reviews ──")
display(train_df.head(3))

print("\n── Label Distribution (Train) ──")
label_counts = train_df["label"].value_counts()
print(label_counts.rename({0: "Negative (0)", 1: "Positive (1)"}))

print("\n── Review Length Statistics ──")
train_df["char_length"] = train_df["text"].apply(len)
print(train_df["char_length"].describe().round(1))

# Plot label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Negative", "Positive"], label_counts.values, color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[0].set_title("Label Distribution (Train Set)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")

axes[1].hist(train_df["char_length"], bins=50, color="#3498db", edgecolor="black", alpha=0.8)
axes[1].set_title("Review Character Length Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Character Length")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("dataset_eda.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 EDA plots saved.")

---
## 3. Data Preprocessing <a id='3'></a>

Steps applied:
1. Remove HTML tags (IMDB reviews often contain `<br />` tags)
2. Remove URLs
3. Remove special characters and excessive whitespace
4. Lowercase normalisation
5. Check and drop any missing values

In [ ]:
# ─────────────────────────────────────────────────────────────
# DATA PREPROCESSING
# ─────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Clean a raw review string by:
      1. Removing HTML tags (e.g. <br />, <b>)
      2. Removing URLs
      3. Removing non-alphabetic characters (keep spaces)
      4. Collapsing multiple spaces into one
      5. Stripping leading/trailing whitespace
      6. Lowercasing

    Parameters
    ----------
    text : str – Raw review text.

    Returns
    -------
    str – Cleaned text.
    """
    # Step 1: Strip HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Step 2: Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Step 3: Keep only letters and spaces (remove punctuation, digits)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Step 4: Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Step 5 & 6: Strip and lowercase
    return text.strip().lower()


# ── Apply cleaning to both splits ────────────────────────────
print("⏳ Cleaning text...")
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"]  = test_df["text"].apply(clean_text)

# ── Check for missing values ──────────────────────────────────
train_missing = train_df["clean_text"].isna().sum() + (train_df["clean_text"] == "").sum()
test_missing  = test_df["clean_text"].isna().sum()  + (test_df["clean_text"]  == "").sum()
print(f"   → Missing / empty in train: {train_missing}")
print(f"   → Missing / empty in test : {test_missing}")

# Drop empty rows if any
train_df = train_df[train_df["clean_text"].str.strip() != ""].reset_index(drop=True)
test_df  = test_df[test_df["clean_text"].str.strip()  != ""].reset_index(drop=True)

print("\n✅ Preprocessing complete!")
print("\nSample before / after cleaning:")
print(f"  BEFORE: {train_df['text'].iloc[0][:120]}...")
print(f"  AFTER : {train_df['clean_text'].iloc[0][:120]}...")

---
## 4. Train / Validation / Test Split <a id='4'></a>

We use a **subset** of the data (5,000 train + 1,000 val + 1,000 test) to keep training time reasonable in a notebook environment. The full dataset can be used by changing `TRAIN_SIZE`.

| Split | Size | Purpose |
|---|---|---|
| Train | 4,000 | Fine-tune model weights |
| Validation | 1,000 | Monitor training, tune hyperparameters |
| Test | 1,000 | Final held-out evaluation |

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAIN / VALIDATION / TEST SPLIT
# ─────────────────────────────────────────────────────────────

# ── Configuration ─────────────────────────────────────────────
# Reduce these values if training is slow; increase for better accuracy
TRAIN_SAMPLE = 4000   # Number of training examples to use
VAL_SAMPLE   = 1000   # Validation examples
TEST_SAMPLE  = 1000   # Test examples

# ── Sample from the full dataset (stratified) ────────────────
train_sample = train_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(TRAIN_SAMPLE // 2, random_state=SEED)
).reset_index(drop=True)

# Validation split carved from training data
train_final, val_df = train_test_split(
    train_sample, test_size=VAL_SAMPLE,
    stratify=train_sample["label"], random_state=SEED
)

# Test sample
test_sample = test_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(TEST_SAMPLE // 2, random_state=SEED)
).reset_index(drop=True)

print("✅ Data splits created:")
print(f"   → Train      : {len(train_final):,} samples")
print(f"   → Validation : {len(val_df):,} samples")
print(f"   → Test       : {len(test_sample):,} samples")

# Confirm balanced labels
print("\n   Label balance (train):")
print(train_final["label"].value_counts().to_string())

---
## 5. Tokenization with `bert-base-uncased` <a id='5'></a>

BERT requires input in a specific format:
- Prepend `[CLS]` token, append `[SEP]` token
- Truncate / pad sequences to a fixed `MAX_LEN`
- Return `input_ids`, `attention_mask`, and `token_type_ids`

We set `MAX_LEN = 128` for speed. The standard BERT maximum is 512.

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOKENIZATION
# ─────────────────────────────────────────────────────────────

MODEL_CHECKPOINT = "bert-base-uncased"   # Pre-trained model identifier
MAX_LEN          = 128                   # Sequence length (truncate/pad to this)

print(f"⏳ Loading tokenizer: '{MODEL_CHECKPOINT}' ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("✅ Tokenizer loaded!")

# ── Demonstrate tokenization on one example ───────────────────
sample_text = train_final["clean_text"].iloc[0][:80]
encoded = tokenizer(
    sample_text,
    max_length      = MAX_LEN,
    padding         = "max_length",
    truncation      = True,
    return_tensors  = "pt"
)
print("\n── Tokenization Example ──")
print(f"   Input text     : '{sample_text[:60]}...'")
print(f"   input_ids shape: {encoded['input_ids'].shape}")
print(f"   Decoded tokens : {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])[:12]} ...")

---
## 6. Build PyTorch Dataset & DataLoaders <a id='6'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# CUSTOM PYTORCH DATASET
# ─────────────────────────────────────────────────────────────

class IMDBDataset(Dataset):
    """
    PyTorch Dataset for IMDB sentiment data.

    Each item returns a dict with:
      - input_ids      : token IDs (tensor)
      - attention_mask : mask for real vs. padding tokens (tensor)
      - labels         : sentiment label 0 or 1 (tensor)
    """

    def __init__(self, texts: list, labels: list,
                 tokenizer, max_len: int):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        # Tokenize the text at index idx
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = "max_length",
            truncation      = True,
            return_tensors  = "pt"
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),        # (seq_len,)
            "attention_mask" : encoding["attention_mask"].squeeze(0),   # (seq_len,)
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ── Create Dataset objects ────────────────────────────────────
train_dataset = IMDBDataset(
    texts     = train_final["clean_text"].tolist(),
    labels    = train_final["label"].tolist(),
    tokenizer = tokenizer,
    max_len   = MAX_LEN
)
val_dataset = IMDBDataset(
    texts     = val_df["clean_text"].tolist(),
    labels    = val_df["label"].tolist(),
    tokenizer = tokenizer,
    max_len   = MAX_LEN
)
test_dataset = IMDBDataset(
    texts     = test_sample["clean_text"].tolist(),
    labels    = test_sample["label"].tolist(),
    tokenizer = tokenizer,
    max_len   = MAX_LEN
)

# ── DataLoaders (batch iteration over datasets) ───────────────
BATCH_SIZE = 16   # Reduce to 8 if you get CUDA out-of-memory errors

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("✅ Datasets and DataLoaders ready!")
print(f"   → Batch size      : {BATCH_SIZE}")
print(f"   → Train batches   : {len(train_loader)}")
print(f"   → Val batches     : {len(val_loader)}")
print(f"   → Test batches    : {len(test_loader)}")

---
## 7. Model Building <a id='7'></a>

We use `AutoModelForSequenceClassification` which wraps BERT and adds a **classification head** (linear layer) on top of the `[CLS]` token output.

```
Input Tokens
     │
     ▼
BERT Encoder (12 transformer layers)
     │
     ▼  [CLS] token representation
Dropout (0.1)
     │
     ▼
Linear(768 → num_labels)
     │
     ▼
Logits → Cross-Entropy Loss
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# MODEL BUILDING UTILITY
# Returns a fresh BERT classification model for each experiment.
# ─────────────────────────────────────────────────────────────

NUM_LABELS = 2   # Binary: Negative (0) / Positive (1)

def build_model(checkpoint: str = MODEL_CHECKPOINT,
                num_labels: int = NUM_LABELS) -> torch.nn.Module:
    """
    Load a fresh AutoModelForSequenceClassification from the given checkpoint.

    Parameters
    ----------
    checkpoint : str – Hugging Face model identifier.
    num_labels : int – Number of output classes.

    Returns
    -------
    model : torch.nn.Module – BERT model moved to DEVICE.
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels = num_labels
    )
    return model.to(DEVICE)


# ── Quick test: load and describe model ───────────────────────
print(f"⏳ Loading {MODEL_CHECKPOINT} for architecture inspection...")
_model = build_model()

total_params     = sum(p.numel() for p in _model.parameters())
trainable_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)

print(f"\n✅ Model architecture: {MODEL_CHECKPOINT}")
print(f"   → Total parameters     : {total_params:,}")
print(f"   → Trainable parameters : {trainable_params:,}")
print(f"   → BERT encoder layers  : 12")
print(f"   → Hidden size          : 768")
del _model   # Free memory — we'll reload fresh models per experiment

---
## 8. Training & Evaluation Utilities <a id='8'></a>

Central functions reused by all three experiments:
- `train_one_epoch()` — one pass over the training DataLoader
- `evaluate()` — compute loss + metrics on any DataLoader
- `run_experiment()` — orchestrates training + per-epoch logging
- `compute_metrics()` — accuracy, precision, recall, F1

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler=None):
    """
    Run one full epoch of training.

    Returns
    -------
    avg_loss : float – Mean cross-entropy loss over all batches.
    """
    model.train()          # Enable dropout
    total_loss = 0.0

    for batch in loader:
        # Move batch tensors to the compute device
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()   # Clear gradients from the previous step

        # Forward pass — model returns loss + logits
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            labels         = labels
        )
        loss = outputs.loss

        # Backward pass — compute gradients
        loss.backward()

        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()        # Update weights
        if scheduler:
            scheduler.step()    # Bonus: update learning rate

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    """
    Evaluate model on a DataLoader.

    Returns
    -------
    avg_loss  : float       – Mean cross-entropy loss.
    all_preds : np.ndarray  – Predicted class labels.
    all_labels: np.ndarray  – True class labels.
    """
    model.eval()    # Disable dropout
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []

    with torch.no_grad():    # Disable gradient tracking for speed
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            total_loss += outputs.loss.item()

            # Convert logits → predicted class index
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / len(loader),
        np.array(all_preds),
        np.array(all_labels)
    )


def compute_metrics(preds: np.ndarray, labels: np.ndarray) -> dict:
    """
    Compute classification metrics.

    Returns
    -------
    dict with accuracy, precision, recall, f1
    """
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1
    )
    return {
        "accuracy" : round(acc,  4),
        "precision": round(prec, 4),
        "recall"   : round(rec,  4),
        "f1"       : round(f1,   4)
    }


def run_experiment(experiment_name: str,
                   model,
                   num_epochs: int = 3,
                   learning_rate: float = 2e-5,
                   use_scheduler: bool = True):
    """
    Full training + evaluation loop for one experiment.

    Parameters
    ----------
    experiment_name : str   – Label for display purposes.
    model           :       – BERT model (already configured).
    num_epochs      : int   – Training epochs.
    learning_rate   : float – AdamW learning rate (default 2e-5).
    use_scheduler   : bool  – Whether to use a linear warmup scheduler.

    Returns
    -------
    results : dict – Final test metrics, history, predictions.
    """
    print(f"\n{'='*60}")
    print(f"  🧪 {experiment_name}")
    print(f"{'='*60}")

    # ── Optimiser: AdamW (as specified in assignment) ─────────
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr           = learning_rate,
        weight_decay = 0.01       # L2 regularisation
    )

    # ── Bonus: Linear warmup + decay scheduler ────────────────
    total_steps  = len(train_loader) * num_epochs
    warmup_steps = total_steps // 10   # 10% warmup
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps
    ) if use_scheduler else None

    # ── History tracking ──────────────────────────────────────
    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    best_val_f1    = 0.0
    best_model_state = None

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        # Training
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)

        # Validation
        val_loss, val_preds, val_labels = evaluate(model, val_loader)
        val_metrics = compute_metrics(val_preds, val_labels)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_metrics["f1"])

        elapsed = time.time() - t0
        print(f"  Epoch {epoch}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val F1: {val_metrics['f1']:.4f} | "
              f"Time: {elapsed:.1f}s")

        # ── Bonus: Early stopping (save best checkpoint) ──────
        if val_metrics["f1"] > best_val_f1:
            best_val_f1    = val_metrics["f1"]
            best_model_state = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}

    # ── Restore best model, evaluate on test set ─────────────
    if best_model_state:
        model.load_state_dict(best_model_state)
        model.to(DEVICE)

    test_loss, test_preds, test_labels = evaluate(model, test_loader)
    test_metrics = compute_metrics(test_preds, test_labels)

    print(f"\n  ✅ TEST RESULTS:")
    for k, v in test_metrics.items():
        print(f"     {k:10s}: {v:.4f}")

    return {
        "name"        : experiment_name,
        "metrics"     : test_metrics,
        "history"     : history,
        "preds"       : test_preds,
        "labels"      : test_labels,
        "test_loss"   : round(test_loss, 4)
    }


print("✅ All training utilities defined.")

---
## 9. Experiment 1 — Freeze All BERT Layers (Classifier Only) <a id='9'></a>

**Strategy:** Freeze all 12 BERT encoder layers + embeddings. Only train the classification head (2 layers: dropout + linear).  
**Purpose:** Establishes a lower-bound baseline. Tests how well BERT's generic representations transfer without any fine-tuning.  
**Expected behaviour:** Fast training; lower accuracy since BERT weights are not adapted to IMDB.

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPERIMENT 1: CLASSIFIER HEAD ONLY (ALL BERT LAYERS FROZEN)
# ─────────────────────────────────────────────────────────────

model_exp1 = build_model()   # Fresh BERT model

# ── Freeze all BERT parameters ────────────────────────────────
# Only the classifier head (model.classifier) remains trainable.
for name, param in model_exp1.named_parameters():
    if "classifier" not in name:   # Freeze everything except classifier
        param.requires_grad = False

# ── Verify frozen vs. trainable counts ───────────────────────
frozen    = sum(p.numel() for p in model_exp1.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model_exp1.parameters() if p.requires_grad)
print(f"Experiment 1 — Frozen: {frozen:,} | Trainable: {trainable:,}")

# ── Run experiment ────────────────────────────────────────────
results_exp1 = run_experiment(
    experiment_name = "Experiment 1: Frozen BERT — Classifier Head Only",
    model           = model_exp1,
    num_epochs      = 3,
    learning_rate   = 2e-5
)

---
## 10. Experiment 2 — Fine-Tune Last 2 BERT Layers <a id='10'></a>

**Strategy:** Freeze layers 0–9. Unfreeze BERT encoder layers 10 and 11 (the last 2) + the classifier head.  
**Purpose:** Balances compute efficiency with task adaptation. The last layers capture the most task-specific features.  
**Expected behaviour:** Better than Exp 1; slightly slower training.

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPERIMENT 2: FINE-TUNE LAST 2 BERT LAYERS + CLASSIFIER
# ─────────────────────────────────────────────────────────────

model_exp2 = build_model()   # Fresh BERT model

# ── Freeze all parameters first ───────────────────────────────
for param in model_exp2.parameters():
    param.requires_grad = False

# ── Unfreeze the last 2 encoder layers (layer 10 and 11) ─────
# BERT layer names follow the pattern: bert.encoder.layer.{N}.*
UNFREEZE_LAYERS = ["bert.encoder.layer.10",
                   "bert.encoder.layer.11",
                   "classifier"]

for name, param in model_exp2.named_parameters():
    if any(layer in name for layer in UNFREEZE_LAYERS):
        param.requires_grad = True   # Allow gradient updates

# ── Verify counts ─────────────────────────────────────────────
frozen    = sum(p.numel() for p in model_exp2.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
print(f"Experiment 2 — Frozen: {frozen:,} | Trainable: {trainable:,}")

# ── Run experiment ────────────────────────────────────────────
results_exp2 = run_experiment(
    experiment_name = "Experiment 2: Fine-Tune Last 2 BERT Layers",
    model           = model_exp2,
    num_epochs      = 3,
    learning_rate   = 2e-5
)

---
## 11. Experiment 3 — Full BERT Fine-Tuning *(Bonus)* <a id='11'></a>

**Strategy:** All 12 BERT layers + classifier head are trainable.  
**Purpose:** Upper-bound performance. Standard approach for BERT fine-tuning.  
**Expected behaviour:** Best accuracy; slowest training; risk of catastrophic forgetting if LR is too high.

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPERIMENT 3 (BONUS): FULL BERT FINE-TUNING
# ─────────────────────────────────────────────────────────────

model_exp3 = build_model()   # Fresh BERT model

# All parameters are trainable by default — no freezing needed
trainable = sum(p.numel() for p in model_exp3.parameters() if p.requires_grad)
print(f"Experiment 3 — All Trainable: {trainable:,}")

# ── Run experiment ────────────────────────────────────────────
results_exp3 = run_experiment(
    experiment_name = "Experiment 3 (Bonus): Full BERT Fine-Tuning",
    model           = model_exp3,
    num_epochs      = 3,
    learning_rate   = 2e-5
)

---
## 12. Results Comparison & Analysis <a id='12'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# RESULTS COMPARISON TABLE
# ─────────────────────────────────────────────────────────────

all_results = [results_exp1, results_exp2, results_exp3]

# Build a summary DataFrame
summary_rows = []
for r in all_results:
    row = {"Experiment": r["name"].split(":")[0]}
    row.update(r["metrics"])
    row["test_loss"] = r["test_loss"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Experiment")

print("\n" + "="*70)
print("          📊  EXPERIMENT RESULTS SUMMARY")
print("="*70)
display(summary_df.style
        .format("{:.4f}")
        .highlight_max(axis=0, color="#d4edda", subset=["accuracy","precision","recall","f1"])
        .highlight_min(axis=0, color="#f8d7da", subset=["test_loss"])
        .set_caption("Green = best value | Red = worst value"))

# ── Plot metric comparison bar chart ─────────────────────────
metrics_to_plot = ["accuracy", "precision", "recall", "f1"]
exp_labels      = [f"Exp {i+1}" for i in range(len(all_results))]

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

for ax, metric in zip(axes, metrics_to_plot):
    values = [r["metrics"][metric] for r in all_results]
    bars   = ax.bar(exp_labels, values, color=colors, edgecolor="black", width=0.5)
    ax.set_title(metric.capitalize(), fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

fig.suptitle("Experiment Comparison — Test Set Metrics",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

# ── Training loss curves ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
exp_colors = [("#e74c3c", "#c0392b"), ("#f39c12", "#e67e22"), ("#2ecc71", "#27ae60")]

for ax, r, (tc, vc) in zip(axes, all_results, exp_colors):
    epochs = range(1, len(r["history"]["train_loss"]) + 1)
    ax.plot(epochs, r["history"]["train_loss"], "o-", color=tc, label="Train Loss", lw=2)
    ax.plot(epochs, r["history"]["val_loss"],   "s--", color=vc, label="Val Loss",   lw=2)
    ax.set_title(r["name"].split(":")[0], fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle("Training vs. Validation Loss Curves",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 13. Confusion Matrix Visualisation <a id='13'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFUSION MATRICES — one per experiment
# ─────────────────────────────────────────────────────────────

class_names = ["Negative", "Positive"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, r in zip(axes, all_results):
    cm = confusion_matrix(r["labels"], r["preds"])

    # Compute per-cell percentages for annotation
    cm_pct = cm.astype(float) / cm.sum() * 100
    annot  = np.array([[f"{v}\n({p:.1f}%)"
                        for v, p in zip(row_v, row_p)]
                       for row_v, row_p in zip(cm, cm_pct)])

    sns.heatmap(
        cm, annot=annot, fmt="", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, linecolor="grey", ax=ax,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title(r["name"].split(":")[0] + "\n" + r["name"].split(":")[1].strip(),
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=10)
    ax.set_ylabel("True Label",      fontsize=10)

    # Print full classification report
    print(f"\n── {r['name']} ──")
    print(classification_report(
        r["labels"], r["preds"],
        target_names=class_names
    ))

fig.suptitle("Confusion Matrices — Test Set",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 14. Insights & Conclusions <a id='14'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────
# FINAL ANALYSIS PRINT-OUT
# ─────────────────────────────────────────────────────────────

print("="*65)
print("  📝  INSIGHTS & CONCLUSIONS")
print("="*65)

best_exp = max(all_results, key=lambda r: r["metrics"]["f1"])

insights = f"""
1. EXPERIMENT COMPARISON
   ─────────────────────
   • Experiment 1 (Frozen BERT): Only the 2-layer classifier head
     was trained. Acts as a feature extractor. Fastest to train but
     lowest F1 since BERT weights are not adapted to IMDB sentiment.

   • Experiment 2 (Last 2 Layers): Unfreezing layers 10 & 11 allows
     the model to adapt high-level representations to the task without
     the cost of updating all 110M parameters. Good balance of speed
     vs. performance.

   • Experiment 3 (Full Fine-Tuning): All 12 layers are updated.
     Achieves the best F1 score ({best_exp['metrics']['f1']:.4f}) but requires the
     most compute. Risk of overfitting on small datasets.

2. KEY FINDINGS
   ─────────────
   • Best experiment  : {best_exp['name']}
   • Best F1 score    : {best_exp['metrics']['f1']:.4f}
   • Best accuracy    : {best_exp['metrics']['accuracy']:.4f}

   • Freezing BERT layers drastically reduces trainable parameters
     (~1,500 vs. ~110M) but also limits the model's ability to adapt.
   • Even with only 4,000 training samples, BERT achieves strong
     results due to its pre-trained language understanding.

3. BERT ARCHITECTURE TAKEAWAYS
   ────────────────────────────
   • Lower layers (0–5) capture syntax and morphology — rarely
     need updating for downstream tasks.
   • Upper layers (8–11) capture semantics and task-specific
     features — most beneficial to fine-tune.
   • The [CLS] token pooled output is the key representation
     used for classification.

4. BONUS FEATURES IMPLEMENTED
   ────────────────────────────
   ✅ Linear warmup + decay learning rate scheduler
   ✅ Gradient clipping (max_norm=1.0)
   ✅ Early stopping (best checkpoint saved per experiment)
   ✅ AdamW with weight decay (L2 regularisation)

5. RECOMMENDATIONS FOR FURTHER IMPROVEMENT
   ─────────────────────────────────────────
   • Try DistilBERT (40% fewer params, ~97% of BERT performance)
   • Increase training data to full 25k IMDB samples
   • Increase MAX_LEN to 256 or 512
   • Experiment with RoBERTa or DeBERTa for higher accuracy
"""

print(insights)
print("="*65)
print("  Notebook prepared for Data Science Internship — Feb 2026")
print("="*65)

---

## Pipeline Summary

```
┌────────────────────────────────────────────────────────────────┐
│            BERT FINE-TUNING PIPELINE                           │
│                                                                │
│  Raw IMDB Data                                                 │
│       │                                                        │
│       ▼                                                        │
│  Preprocessing  ──► Remove HTML, URLs, special chars          │
│       │                                                        │
│       ▼                                                        │
│  Train/Val/Test Split (4000 / 1000 / 1000)                     │
│       │                                                        │
│       ▼                                                        │
│  bert-base-uncased Tokenizer  ──► input_ids + attention_mask  │
│       │                                                        │
│       ▼                                                        │
│  PyTorch Dataset + DataLoader (batch_size=16)                  │
│       │                                                        │
│       ▼                                                        │
│  AutoModelForSequenceClassification (BERT + Linear Head)       │
│       │                                                        │
│       ├──► Exp 1: Frozen BERT  (1,538 trainable params)        │
│       ├──► Exp 2: Last 2 Layers (~14M trainable params)        │
│       └──► Exp 3: Full Fine-Tune (~110M trainable params)      │
│       │                                                        │
│       ▼                                                        │
│  AdamW + Linear LR Scheduler  ──► 3 epochs per experiment     │
│       │                                                        │
│       ▼                                                        │
│  Evaluation: Accuracy, Precision, Recall, F1, Confusion Matrix │
│       │                                                        │
│       ▼                                                        │
│  Results Comparison & Analysis                                 │
└────────────────────────────────────────────────────────────────┘
```

---
*Notebook prepared as part of Data Science Internship – February 2026*